In [4]:
# Load environment variables from .env file
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [4]:
from agents import Agent, Runner

# Create an agent with a name and instructions
agent = Agent(
    name="My First Agent", 
    instructions="To demonstrate the OpenAI Agents SDK",
    model="gpt-5-nano-2025-08-07"
)
# Create a runner to execute the agent
runner = Runner()
# Run the agent
result = await runner.run(agent, 'Tell me a joke about coffee.')
print(result.final_output)

Why did the coffee file a police report? It got mugged.


In [9]:
from agents import Agent, Runner

agent = Agent(
    name="Trace Stack Analyzer",
    instructions=f"""
You are an expert API test validation engine.
Your task is to analyze API test steps and determine whether the status code check step needs to be updated.
Follow these rules strictly:
1. Identify the endpoint call step and extract:
 The request details
 The actual response status code
 The response body
2. Identify the step that checks the status code and extract:
  The expected status code
3. The expected status code must always be validated against:
  Swagger/OpenAPI documentation (if provided)
  The intended test scenario (validation error, auth error, success case, etc.)
4. Before deciding to update the expected status code, verify:
  Whether the test setup is correct
  Whether required authentication/authorization steps are missing
  Whether headers (Authorization, Content-Type, etc.) are properly included
  Whether the failure is due to environment/setup issues rather than business logic
5. If the API returned a different status code due to:
  Missing authentication
  Missing required headers
  Incorrect test setup
  Environment or access issues
    → Then DO NOT update the expected status code.
    → The correct result is "NO_UPDATE" because the test case itself is invalid.
6. Only return "UPDATE" when:
 The test setup is valid
 The request is correctly formed
 Authentication (if required) is properly handled
 The actual API behavior consistently differs from Swagger documentation
 And the API specification has changed or Swagger is outdated
7. Never update the expected status code just because the actual response differs.
 The root cause must be analyzed first.

Return the result strictly in one of the following formats:

If no update is needed:
{{"status": "NO_UPDATE"}}

If update is required:
{{"status": "UPDATE", "new_status_code": "<status_code>", "reason": "<clear root cause explanation>"}}

Do not include any extra text outside the JSON response.
    """,
    model="gpt-5-nano-2025-08-07"
)

test_steps = """
"with body \"{\"petId\": 8447; \"quantity\": 2883; \"id\": 9090; \"shipDate\": \"2023-12-01T10:00:00Z\"; \"status\": \"INVALID_ENUM_VALUE\"}\", Status: Pass, Execution Time: 0.00s, Output: <end>\ncall post \"store/order\", Status: Pass, Execution Time: 0.00s, Output: Request response: {\"timestamp\":\"2026-02-19T12:05:21.367+00:00\",\"status\":403,\"error\":\"Forbidden\",\"message\":\"Access Denied\",\"path\":\"/store/order\"}<end>\ncheck status code is 400, Status: Fail, Execution Time: 0.00s, Output: Expected: 400, Actual: 403<end>\n",
"""
runner = Runner()
result = await runner.run(agent, test_steps)
print(result.final_output)

{"status": "NO_UPDATE"}
